# Reproducing DocLens (ACL 2026) on Kaggle

This notebook reproduces the main results of *DocLens: A Tool-Augmented Multi-Agent Framework for Long Visual
Document Understanding* with the **official, unmodified DocLens code**
(https://github.com/dwzhu-pku/DocLens):

1. the **full** benchmark (all questions of MMLongBench-Doc, or FinRAGBench-V),
2. **DocLens end-to-end** (Page Navigator → Element Localizer → Answer Sampler → Adjudicator),
3. the **vanilla baseline** (the same model reading every page once), as in the paper's tables,
4. the paper's metrics: MMLongBench-Doc accuracy, F1, and the TXT/LAY/CHA/TAB/IMG and SIN/MUL/UNA breakdowns.

The runner adds only engineering around DocLens (checkpointing, rate limiting, API-key or Vertex AI client).
Agents, prompts, page rendering, MinerU parsing and scoring are DocLens' own.

### What "same results as the paper" requires
* **The same models**: Gemini 2.5 Pro (answering), 2.5 Flash-Lite (page navigation), 2.5 Flash (answer extraction for scoring).
  These are no longer offered to *new* Gemini API keys. **Google Cloud Vertex AI** (DocLens' original setup) is the most likely
  route to them. Section 3 tests which models your account can actually call.
* **The same settings**: the number of sampled candidates for the Page Navigator and Answer Sampler. The repository's scripts disagree
  with each other, so **copy them from the paper's implementation details** into the settings cell.
* **Small differences are expected.** DocLens samples at temperature 0.7, so a faithful reproduction lands within about 1–2 points, not on the exact digits.

### Before running
* *Settings → Accelerator*: **GPU T4 x2** · *Settings → Internet*: **On**.
* *Add-ons → Secrets*, then either:
  * **Vertex AI** (recommended): `GCP_SA_KEY` = the full JSON of a Google Cloud service-account key with the *Vertex AI User* role; or
  * **API key**: `GEMINI_API_KEY` = a Google AI Studio key.
* Budget: the full run makes several thousand model calls. Run the **pilot** (section 7) first and check your billing page.

In [ ]:
# ======================= SETTINGS =======================
WORK = "/kaggle/working"
CDC_REPO = "https://github.com/Mariam1543/aqqal123.git"
CDC_BRANCH = "claude/wizardly-rubin-kdthhy"

DATASET = "MMLongBenchDoc"          # or "FinRAGBench-V"
ACCESS = "vertex"                   # "vertex" (original DocLens setup) or "api_key"
GCP_LOCATION = "us-east1"           # DocLens' default Vertex location

# Models used in the DocLens paper (main results). Change only if section 3 shows they are unavailable.
MODEL = "gemini-2.5-pro"            # Answer Sampler, Adjudicator, and the vanilla baseline
NAVIGATOR_MODEL = "gemini-2.5-flash-lite"   # Page Navigator (hard-coded in DocLens' main.py)
EVAL_MODEL = "gemini-2.5-flash"     # answer extraction in DocLens' evaluation (hard-coded in eval_toolkits.py)

# !!! Copy these from the paper's implementation details !!!
# The repo's run_doclens_end2end.sh uses 2 and 2; run_doclens_phase1.sh / phase2.sh use 8.
PHASE1_K = 2                        # Page Navigator samples
PHASE2_K = 2                        # Answer Sampler candidates
# Temperature 0.7 and 200-DPI page images are DocLens defaults and are kept.

RUN_BASELINE = True                 # also run the vanilla baseline row of the paper's table
PILOT_QUESTIONS = 20
RPM = 60                            # requests/minute cap; raise for high quotas
MAX_CONCURRENT = 8

## 1. Get the code

In [ ]:
import os, subprocess, shutil, json, glob
os.chdir(WORK)
if not os.path.exists("DocLens"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/dwzhu-pku/DocLens.git"], check=True)
if not os.path.exists("doclens_cdc/scripts/report_reproduction.py"):
    shutil.rmtree("doclens_cdc", ignore_errors=True)
    shutil.rmtree("_cdc_repo", ignore_errors=True)
    subprocess.run(["git", "clone", "--depth", "1", "-b", CDC_BRANCH, CDC_REPO, "_cdc_repo"], check=True)
    shutil.copytree("_cdc_repo/doclens_cdc", "doclens_cdc")
    shutil.rmtree("_cdc_repo")

DOCLENS, CDC = f"{WORK}/DocLens", f"{WORK}/doclens_cdc"
ROOT = f"{WORK}/full"               # rendered + parsed benchmark (reused later for the novelty experiments)
RESULTS = f"{WORK}/results_repro"
os.makedirs(RESULTS, exist_ok=True)
print(subprocess.run(["git", "-C", "DocLens", "log", "-1", "--format=DocLens commit %h (%cd)"],
                     capture_output=True, text=True).stdout)

## 2. Install
MinerU **2.x** is required (MinerU 3/4 replaced the command-line interface DocLens uses).

In [ ]:
!pip install -q "mineru[core]==2.7.6" 2>&1 | tail -n 2
!pip install -q -r {CDC}/requirements-kaggle.txt 2>&1 | tail -n 2
!mineru --version

## 3. Credentials and model availability
This decides whether the run is a **reproduction** (the paper's models) or a **re-run with substitute models**.

In [ ]:
from kaggle_secrets import UserSecretsClient
from google import genai
secrets = UserSecretsClient()

if ACCESS == "vertex":
    key_json = secrets.get_secret("GCP_SA_KEY")
    key_path = "/tmp/gcp_key.json"          # outside /kaggle/working so it is never saved with the outputs
    with open(key_path, "w") as f:
        f.write(key_json)
    os.chmod(key_path, 0o600)
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = key_path
    os.environ["GOOGLE_CLOUD_PROJECT"] = json.loads(key_json)["project_id"]
    os.environ["GOOGLE_CLOUD_LOCATION"] = GCP_LOCATION
    client = genai.Client(vertexai=True, project=os.environ["GOOGLE_CLOUD_PROJECT"], location=GCP_LOCATION)
    ACCESS_ARGS = "--use_vertex"
else:
    os.environ["GEMINI_API_KEY"] = secrets.get_secret("GEMINI_API_KEY")
    client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
    ACCESS_ARGS = ""

PAPER_MODELS = {"MODEL": "gemini-2.5-pro", "NAVIGATOR_MODEL": "gemini-2.5-flash-lite", "EVAL_MODEL": "gemini-2.5-flash"}
available = {}
for role, name in {"MODEL": MODEL, "NAVIGATOR_MODEL": NAVIGATOR_MODEL, "EVAL_MODEL": EVAL_MODEL}.items():
    try:
        client.models.generate_content(model=name, contents="Reply with OK.")
        available[role] = True
        print(f"  OK       {role:16s} {name}")
    except Exception as e:
        available[role] = False
        print(f"  FAILED   {role:16s} {name}: {str(e)[:160]}")

SAME_MODELS = all(available.values()) and all(
    {"MODEL": MODEL, "NAVIGATOR_MODEL": NAVIGATOR_MODEL, "EVAL_MODEL": EVAL_MODEL}[k] == v for k, v in PAPER_MODELS.items())
print()
if not all(available.values()):
    print("STOP: a configured model is not callable. Fix access or change the model names in the settings cell.")
elif SAME_MODELS:
    print("All three paper models are available: this run is a REPRODUCTION.")
else:
    print("Substitute models in use: report this run as 'DocLens re-run with <models>', NOT as a reproduction.")

## 4. Download the benchmark

In [ ]:
!python {CDC}/scripts/download_data.py --dataset {DATASET} --out {DOCLENS}/data
SAMPLES = f"{DOCLENS}/data/{DATASET}/samples.json"
DOCUMENTS = f"{DOCLENS}/data/{DATASET}/documents"
assert os.path.exists(SAMPLES), glob.glob(f"{DOCLENS}/data/{DATASET}/**/*.json", recursive=True)[:10]
N_EXPECTED = len(json.load(open(SAMPLES)))
print(N_EXPECTED, "questions;", len(glob.glob(f"{DOCUMENTS}/*.pdf")), "PDFs")

## 5. Render every page at 200 DPI (DocLens' setting)

In [ ]:
!python {CDC}/scripts/prepare_subset.py --all --samples {SAMPLES} --documents {DOCUMENTS} --out_root {ROOT} \
    --dataset_name {DATASET} --workers 4
QUESTIONS = f"{ROOT}/samples_subset.json"
!du -sh {ROOT}

## 6. Parse every page with MinerU (GPU, a few hours)
This is DocLens' Element Localizer input. It **resumes** where it stopped if the session ends: just run the cell again.

**Save this work.** When the cell finishes, click *Save Version → Quick Save*. In later sessions, add this notebook's output as an input
(*Add Input → Your Work → this notebook*) and copy `full/` back to `/kaggle/working`, so you never have to parse again.

In [ ]:
!python {CDC}/scripts/parse_mineru.py --root {ROOT}/clean --dataset_name {DATASET} --batch_size 400
n_pages = len(glob.glob(f"{ROOT}/clean/data/{DATASET}/documents/*/*.jpeg"))
n_md = len(glob.glob(f"{ROOT}/clean/data/{DATASET}/documents/*/MinerU_Page*/*.md"))
print(f"{n_md}/{n_pages} pages parsed")
!du -sh {ROOT}

## 7. Pilot (20 questions): check that everything works and estimate the cost

In [ ]:
run = (f"python {CDC}/run_cdc.py --doclens_root {DOCLENS} --data_root {ROOT}/clean --variant clean "
       f"--dataset_name {DATASET} --samples {QUESTIONS} --model_name {MODEL} --navigator_model {NAVIGATOR_MODEL} "
       f"--eval_model {EVAL_MODEL} --phase1_candidate_num {PHASE1_K} --phase2_candidate_num {PHASE2_K} "
       f"--rpm {RPM} --max_concurrent {MAX_CONCURRENT} {ACCESS_ARGS}")
!{run} --mode doclens --limit {PILOT_QUESTIONS} --output {RESULTS}/pilot_doclens.jsonl

In [ ]:
rows = [json.loads(l) for l in open(f"{RESULTS}/pilot_doclens.jsonl")]
ok = [r for r in rows if "final_score" in r]
print(f"{len(ok)}/{len(rows)} scored, accuracy {sum(r['final_score'] for r in ok) / max(1, len(ok)):.3f}")
tok = sum(r.get("final_tokens", 0) for r in ok) / max(1, len(ok))
print(f"~{tok:,.0f} tokens per question (pipeline, excluding scoring) -> ~{tok * N_EXPECTED / 1e6:,.0f}M tokens for the full benchmark")
for r in rows:
    if "cdc_error" in r:
        print("ERROR:", r["cdc_error"][:400]); break

## 8. DocLens end-to-end on the full benchmark
Resumable: re-run the cell after an interruption and it continues from the checkpoint file.

In [ ]:
!{run} --mode doclens --output {RESULTS}/doclens_full.jsonl

## 9. Vanilla baseline on the full benchmark (the paper's 'without DocLens' row)

In [ ]:
if RUN_BASELINE:
    !{run} --mode baseline --output {RESULTS}/baseline_full.jsonl

## 10. Compare with the paper
Fill in `PAPER` with the numbers from the paper's main table (in percent) for the rows you ran.
Leave a metric out if you are not sure of it.

In [ ]:
PAPER = {
    # f"{MODEL} + DocLens": {"Acc": None, "F1": None, "UNA": None},
    # f"{MODEL} (vanilla)":  {"Acc": None, "F1": None},
}
PAPER = {k: {m: v for m, v in d.items() if v is not None} for k, d in PAPER.items()}
json.dump(PAPER, open(f"{RESULTS}/paper_numbers.json", "w"), indent=2)

runs = [f'--run "{MODEL} + DocLens={RESULTS}/doclens_full.jsonl"']
if RUN_BASELINE and os.path.exists(f"{RESULTS}/baseline_full.jsonl"):
    runs.append(f'--run "{MODEL} (vanilla)={RESULTS}/baseline_full.jsonl"')
RUN_ARGS = " ".join(runs)
!python {CDC}/scripts/report_reproduction.py {RUN_ARGS} --dataset_name {DATASET} \
    --expected_questions {N_EXPECTED} --paper {RESULTS}/paper_numbers.json --out {RESULTS}/reproduction.md
print("\nSame models as the paper:", SAME_MODELS, "| PHASE1_K =", PHASE1_K, "| PHASE2_K =", PHASE2_K)

## 11. Next: your novelty on the same data
The rendered and parsed benchmark in `full/clean` is the starting point for the confidence-gated abstention experiments.
Create degraded copies of it and run the CDC analysis with the **same models and K** as above:

```
!python {CDC}/scripts/make_variants.py --out_root {ROOT} --dataset_name {DATASET} --variants blur_s2 noise_s2 jpeg_s3 lowres_s2 mixed_s2
# parse each variant with scripts/parse_mineru.py, then for each variant:
!{run} --data_root {ROOT}/<variant> --variant <variant> --mode analyze --output {RESULTS}/<variant>_analyze.jsonl
```
Running `--mode analyze` on `full/clean` as well checks that your extension does not hurt clean documents.